In [ ]:
!pip install mlphon

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.6/371.6 kB 5.9 MB/s eta 0:00:00
  Created wheel for mlphon: filename=mlphon-3.1.2-py3-none-any.whl size=22128 sha256=a284284372690d822adab687c8de74931555901f56e2be98644ea9e4c47a1b27
  Stored in directory: /root/.cache/pip/wheels/19/7f/bd/7506f80aa81b33e1703ec5ee42b8d856f78700a675d4691d54
Successfully built mlphon


In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

Saving malayalam.txt to malayalam.txt
Uploaded file: malayalam.txt


In [ ]:
import re
from mlphon import PhoneticAnalyser

mlphon = PhoneticAnalyser()

# Define stress markers based on Malayalam long vowels
def is_stressed(syllable):
    # Long vowel markers
    stress_markers = ['ാ', 'ീ', 'ൂ', 'േ', 'ൈ', 'ോ', 'ൌ']

    # Entire syllable is a long vowel (e.g., ആ, ഈ, ഊ)
    stressed_vowels = ['ആ', 'ഈ', 'ഊ', 'ഓ', 'ഏ', 'ഐ', 'ഔ', 'ഏ']

    return any(marker in syllable for marker in stress_markers) or syllable in stressed_vowels

# Preprocess line to remove punctuation and clean weird characters
def clean_line(line):
    return re.sub(r"[^\u0D00-\u0D7F\s]", "", line)  # Keep only Malayalam chars and spaces

def analyze_line(line):
    line = clean_line(line)
    words = line.strip().split()

    syllables = []
    stressed_syllables = []

    for word in words:
        try:
            word_syllables = mlphon.split_to_syllables(word)
            syllables.extend(word_syllables)
            stressed_syllables.extend([syl for syl in word_syllables if is_stressed(syl)])
        except Exception as e:
            print(f"Could not split '{word}' into syllables")

    total = len(syllables)
    stressed = len(stressed_syllables)
    unstressed = total - stressed

    return total, stressed, unstressed, syllables, stressed_syllables

In [ ]:
results = []

with open(filename, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if line.strip():
        try:
            total, stressed, unstressed, syllables, stressed_syllables = analyze_line(line)
            results.append({
                "Line No.": i + 1,
                "Lyrics": line.strip(),
                "Total Syllables": total,
                "Stressed": stressed,
                "Unstressed": unstressed,
                "Syllables": syllables,
                "Stressed Syllables": stressed_syllables
            })
        except Exception as e:
            print(f"Error in line {i + 1}: {e}")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
df[["Line No.", "Lyrics", "Total Syllables", "Stressed", "Unstressed", "Syllables", "Stressed Syllables"]]

,Line No.,Lyrics,Total Syllables,Stressed,Unstressed,Syllables,Stressed Syllables
0,1,ആ ഒരുത്തി അവളൊരുത്തി,9,1,8,"[ആ, ഒ, രു, ത്തി, അ, വ, ളൊ, രു, ത്തി]",[ആ]
1,2,പാൽ മണക്കും കതിരൊരുത്തി,9,1,8,"[പാൽ, മ, ണ, ക്കും, ക, തി, രൊ, രു, ത്തി]",[പാൽ]
2,3,ആ ഒരുത്തി അവളൊരുത്തി,9,1,8,"[ആ, ഒ, രു, ത്തി, അ, വ, ളൊ, രു, ത്തി]",[ആ]
3,4,പാൽ മണക്കും കതിരൊരുത്തി,9,1,8,"[പാൽ, മ, ണ, ക്കും, ക, തി, രൊ, രു, ത്തി]",[പാൽ]
4,5,ഒരുത്തിയന്നെന്നെ പിരിഞ്ഞ നേരത്ത്,12,1,11,"[ഒ, രു, ത്തി, യ, ന്നെ, ന്നെ, പി, രി, ഞ്ഞ, നേ, ര, ത്ത്]",[നേ]
5,6,പൊഴിഞ്ഞതൊക്കെയും പാഴിലാ...,9,2,7,"[പൊ, ഴി, ഞ്ഞ, തൊ, ക്കെ, യും, പാ, ഴി, ലാ]","[പാ, ലാ]"
6,7,കരഞ്ഞു നീലിച്ച കനവിലൊക്കെയും,12,1,11,"[ക, ര, ഞ്ഞു, നീ, ലി, ച്ച, ക, ന, വി, ലൊ, ക്കെ, യും]",[നീ]
7,8,കരിമുകിലിൻ ചാകരാ..,8,2,6,"[ക, രി, മു, കി, ലിൻ, ചാ, ക, രാ]","[ചാ, രാ]"
8,10,ഈ ഒരുത്തി ഇവളൊരുത്തി,9,1,8,"[ഈ, ഒ, രു, ത്തി, ഇ, വ, ളൊ, രു, ത്തി]",[ഈ]
9,11,പാലൊഴുകും ചിരി പരത്തി..,9,1,8,"[പാ, ലൊ, ഴു, കും, ചി, രി, പ, ര, ത്തി]",[പാ]
